# Traffic Speed Visualization
Visualizes road segment average speeds using MapboxGL.

In [22]:
# 1. Install Dependencies (if needed)
# !pip install requests pandas geopandas mapboxgl shapely

In [23]:
# 2. Setup
import requests
import pandas as pd
from mapboxgl.viz import LinestringViz
from mapboxgl.utils import create_color_stops

MAPBOX_TOKEN = "pk.eyJ1IjoicG5lbHNvbnAxIiwiYSI6ImNtbmwzaDZsODE2Z2Myb29kZDQ1ajJmM3cifQ.AqqsdmIk4iOuay55Pp-rGQ"
BASE_URL = "http://localhost:8000"

In [24]:
# 3. Request Aggregated Data
params = {
    "day": "Monday",
    "period": "AM Peak",
    "limit": 1000,
}
response = requests.get(f"{BASE_URL}/aggregates/", params=params)
response.raise_for_status()
geojson_data = response.json()
print(f"Fetched {len(geojson_data)} road segments")

Fetched 1000 road segments


In [25]:
# 4. Visualize in Mapbox
features = [
    {
        "type": "Feature",
        "geometry": f["geometry"],
        "properties": {
            "average_speed": f["average_speed"],
            "road_name": f["road_name"],
        },
    }
    for f in geojson_data
]

viz = LinestringViz(
    {"type": "FeatureCollection", "features": features},
    access_token=MAPBOX_TOKEN,
    color_property="average_speed",
    color_stops=create_color_stops([10, 20, 30, 40, 50], colors="RdYlGn"),
    center=(-81.6557, 30.3322),
    zoom=11,
    line_width_default=1.5,
    opacity=0.8,
)
viz.show()

/home/phyllis/.pyenv/versions/3.12.4/lib/python3.12/site-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


In [26]:
# 5. Tabular Summary — 10 slowest road segments
df = pd.DataFrame([
    {
        "link_id": f["link_id"],
        "avg_speed": f["average_speed"],
        "road_name": f["road_name"],
        "length_m": f["length_m"],
    }
    for f in geojson_data
])
df.sort_values("avg_speed").head(10)

,link_id,avg_speed,road_name,length_m
536,1021821186,0.62,Swelo Rd,16.9
541,1021905649,1.24,Regency Court Mall,20.6
795,1033382080,1.24,None,15.5
673,1032710453,1.24,Mt Herman St,120.5
719,1032971696,1.24,None,87.9
979,1049745903,1.55,Bartram Office Park,83.4
79,1006497018,1.55,Irongate Dr,76.5
740,1032980614,1.71,Saints Rd,76.1
744,1032983340,1.86,Kingsbury St,22.2
916,1046257776,1.86,Grasse St,191.5
